# Evaluasi Tender Insight Agent

Notebook ini menjalankan ekstraksi, checklist, Risk Engine, dan evaluasi akurasi pada tiga dokumen pengumuman tender fiktif di folder `samples/`, terhadap profil perusahaan contoh yang sama. Tanggal analisis dikunci **7 November 2026** agar hasil dapat direproduksi.

In [ ]:
import json
from datetime import date

from extractor import extract
from checklist import build_checklist
from risk import analyze_risk
from cli import analyze_file

profile = json.load(open('samples/company_profile.json', encoding='utf-8'))
analysis_date = date(2026, 11, 7)
print('Profil perusahaan:', profile['nama'])

Profil perusahaan: PT Bangun Contoh Sejahtera


## 1. Hasil ekstraksi satu dokumen

Perhatikan field baru versi 2 (jangka waktu, metode evaluasi, jenis kontrak) dan blok `confidence` per kelompok field.

In [ ]:
gedung = extract(open('samples/tender_gedung.txt', encoding='utf-8').read())
print(json.dumps({k: gedung[k] for k in ['nama_paket','instansi','nilai_hps_rp','jangka_waktu_hari','metode_evaluasi','jenis_kontrak','jadwal','kualifikasi','confidence']}, ensure_ascii=False, indent=2))

{
  "nama_paket": "Pembangunan Gedung Kantor Dinas Terpadu Tahap II",
  "instansi": "Dinas Pekerjaan Umum dan Penataan Ruang Kabupaten Sukamaju",
  "nilai_hps_rp": 12450000000,
  "jangka_waktu_hari": 180,
  "metode_evaluasi": "Sistem Gugur",
  "jenis_kontrak": "Kontrak Harga Satuan",
  "jadwal": {
    "penjelasan": "2026-11-16",
    "batas_pemasukan": "2026-11-27",
    "pembukaan": "2026-11-28",
    "masa_sanggah": "2026-11-30",
    "pengumuman_pemenang": "2026-12-07"
  },
  "kualifikasi": {
    "sbu": [
      "BG002"
    ],
    "pengalaman_min_rp": 8000000000,
    "pengalaman_dalam_tahun": 4,
    "modal_min_rp": 2500000000,
    "tenaga_ahli": [
      "Ahli Teknik Bangunan Gedung (SKA Madya)",
      "Ahli K3 Konstruksi"
    ],
    "peralatan": [
      "Tower Crane",
      "Concrete Pump",
      "Scaffolding"
    ]
  },
  "confidence": {
    "identitas": "tinggi",
    "jadwal": "tinggi",
    "kualifikasi": "tinggi",
    "dokumen": "tinggi",
    "pelaksanaan": "tinggi"
  }
}


## 2. Skor, rekomendasi, dan risiko ketiga dokumen

In [ ]:
for name in ['tender_gedung.txt', 'tender_jalan.txt', 'tender_irigasi.txt']:
    e, r, risk, ins = analyze_file(f'samples/{name}', profile, analysis_date)
    print(name, e['nilai_hps_rp'], r['skor'], r['rekomendasi'], risk['level'], risk['skor_risiko'], len(ins['pertanyaan_aanwijzing']))

berkas                HPS (Rp)      skor  rekomendasi          risiko        pertanyaan
tender_gedung.txt     12.450.000.000    87  BID                  Sedang( 39)  4
tender_jalan.txt      6.875.500.000    80  BID DENGAN CATATAN   Tinggi(100)  5
tender_irigasi.txt    3.275.000.000    42  NO-BID               Sedang( 40)  5


## 3. Evaluasi akurasi ekstraksi

Sembilan field per dokumen dibandingkan dengan `samples/expected.json` (27 field total) oleh `run_eval.py`.

In [ ]:
import subprocess, sys
print(subprocess.run([sys.executable, 'run_eval.py'], capture_output=True, text=True).stdout.splitlines()[-1])

Akurasi pada set contoh: 27/27 field benar (100.0%)


## Kesimpulan

- Ekstraksi deterministik mencapai **27 dari 27 field benar** pada set contoh.
- Tiga dokumen menghasilkan tiga keputusan berbeda: BID (87), BID DENGAN CATATAN (80), NO-BID (42).
- Risk Engine memisahkan risiko dari skor kesiapan: paket jalan mendapat risiko **Tinggi** walau skornya 80, karena jendela persiapannya hanya 2 hari dan dananya DAK.
- Angka pada notebook ini adalah keluaran nyata kode repo, bukan contoh karangan.